In [5]:
"""
Cell Painting Pipeline — generated from Cell_Paint_3.ipynb
All parameters are defined in the CONFIG block at the top.
Edit them here or use the GUI to regenerate.
"""

import os
import re
import sys
from pathlib import Path

from tkinter import FALSE
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from pycytominer import normalize, feature_select
from metadata_utils import parse_value, compute_alpha_per_group, assign_colors



# =============================================================================
# CONFIG — edit all parameters here
# =============================================================================

# if len(sys.argv) < 2:
#     print("Usage: python cell_paint_pipeline.py <experiment_folder> <bool_new>")
#     print("  e.g. python cell_paint_pipeline.py /Volumes/KINGSTON/Nico_data/Cellpaint/20260319 True")
#     sys.exit(1)
 
EXPERIMENT_DIR = Path("/Volumes/KINGSTON/Nico_data/Cellpaint/20260423_HUVEC_TNF").resolve()
ROOT_PATH      = EXPERIMENT_DIR.parent          # e.g. …/Cellpaint/
FILE_NAME      = EXPERIMENT_DIR.name            # e.g. 20260319
 
SQLITE_FILE      = EXPERIMENT_DIR / "cell_profiler_out" / "CellPaint.db"
OUTPUT_CHUNK_DIR = EXPERIMENT_DIR / "output_chunks"
OUTPUT_PLT_DIR   = EXPERIMENT_DIR / "plt_analysis"
OUTPUT_RSLT_DIR  = EXPERIMENT_DIR / "rslt"
FINAL_OUTPUT     = OUTPUT_RSLT_DIR / "final_profiles.csv"
FILTERED_OUTPUT  = OUTPUT_RSLT_DIR / "filtered_profiles.csv"
METADATA_FILE    = "./00_Metadata_Experiments.xlsx"
 
# SELECT_NAMES: derived from the folder name — everything up to the first
# purely-numeric segment is kept, then rejoined.
# e.g. "20260319_HDMVEC" → "20260319_HDMVEC"  (folder name is already the Name)
SELECT_NAMES = FILE_NAME if not FILE_NAME.startswith("20") else FILE_NAME[2:]   # override here if your folder name differs from the Name column
 
CHUNKSIZE  = 100_000
AGG_METHOD = "median"   # "median" or "mean"
BOOL_NEW   = True # True = recompute from SQLite even if CSV already exists
 
#only create output directories if the SQLITE file exists, otherwise we might be pointing to the wrong experiment folder
if not SQLITE_FILE.exists():
    raise FileNotFoundError(f"SQLite file not found at {SQLITE_FILE}. Please check the Path you provided: {EXPERIMENT_DIR}.")

os.makedirs(OUTPUT_CHUNK_DIR, exist_ok=True)
os.makedirs(OUTPUT_PLT_DIR,   exist_ok=True)
os.makedirs(OUTPUT_RSLT_DIR,  exist_ok=True)
 
print(f"Experiment : {EXPERIMENT_DIR}")
print(f"Root       : {ROOT_PATH}")
print(f"SQLite     : {SQLITE_FILE}")
print(f"Metadata   : {METADATA_FILE}")
print(f"Plots  →   : {OUTPUT_PLT_DIR}")
print(f"Results →  : {OUTPUT_RSLT_DIR}")


# --- Metadata & experiment --- Must match "Name" column in Excel
MAX_IMAGES_WARNING   = 864                 # Warn if Per_Image rows exceed this
CELL_COUNT_THRESH    = 3.0                 # Drop images < mean - N*std cell count
CELL_AREA_THRESH     = 2.0                 # Drop images > mean + N*std cell area

# --- Data loading & aggregation ---
DROP_KEYWORDS        = ["COSTES", "PARENT", "CHILD"]  # Feature name exclusions
DROP_OBJECT_NUMBER   = True
DROP_WELL            = True

# --- Normalization ---
NORM_METHOD          = "standardize"     # "mad_robustize", "standardize", "robustize"
MAD_EPS              = 0                  # Epsilon for MAD robustize

# --- Feature selection ---
FEATURE_OPS          = ["variance_threshold", "correlation_threshold", "drop_na_columns"]
CORR_THRESHOLD       = 0.8
OUTLIER_CUTOFF       = 10

# --- Heatmap & treatment clustering ---
SCALER               = "StandardScaler"   # "StandardScaler" or "RobustScaler"
TOP_FEATURES         = 60
HEATMAP_CMAP         = "PiYG"
N_CLUSTERS           = 8
CLUST_METRIC         = "cosine"
CLUST_LINKAGE        = "average"

# --- Dimensionality reduction ---
N_SAMPLE             = 20_000
PCA_N_COMPONENTS     = 60
TSNE_PERPLEXITY      = 100
TSNE_MAX_ITER        = 1000
TSNE_INIT            = "pca"
TSNE_LR              = "auto"
UMAP_N_NEIGHBORS     = 10
UMAP_MIN_DIST        = 0.1
UMAP_METRIC          = "cosine"
RANDOM_SEED          = 42

DO_PCA               = True
DO_TSNE              = True
DO_UMAP              = False

# --- Plot output ---
PLT_DPI              = 450
TRANSPARENT          = True
SCATTER_SIZE         = 5
LEGEND_FONTSIZE      = 10
LEGEND_MARKERSCALE   = 3
ANNOT_FONTSIZE       = 5

PLOT_CLUSTERMAP_V          = True
PLOT_CLUSTERMAP_H          = True
PLOT_SIMILARITY            = True
PLOT_SIMILARITY_CLUSTERED  = False
PLOT_TSNE                  = True
PLOT_TSNE_CLUSTERED        = True
PLOT_TSNE_GRID             = True
PLOT_UMAP                  = True


# =============================================================================
# METADATA
# =============================================================================
print("\n[1/8] Loading metadata...")
metadata = pd.read_excel(METADATA_FILE)
metadata = metadata.dropna(
    subset=["Well_Row", "Well_Column", "Treatment", "Incub time [h]", "Date"]
)
metadata["Image_Metadata_Well"] = (
    metadata["Well_Row"].astype(str) + metadata["Well_Column"].astype(int).astype(str)
)
metadata["treatment_full"] = (
    metadata["Treatment"].astype(str) + "_" +
    metadata["Concentration (ng)"].astype(str) + "_" +
    metadata["Incub time [h]"].astype(str)
)

# Parse date from FILE_NAME
date_str = [s for s in str(FILE_NAME).split("_") if s.isdigit()]
if not date_str:
    raise ValueError(f"No numeric date found in FILE_NAME='{FILE_NAME}'. Expected e.g. '20260319' or '20260319_HAEC'.")
if date_str[0].startswith("20"):
    date = int(date_str[0][2:])  # last 6 digits → YYMMDD
else:
    date = int(date_str[0])       # already YYMMDD
metadata = metadata[metadata["Date"] == date]
# metadata = metadata.loc[metadata["Concentration (ng)"].isna()].fillna(0).copy()

metadata["concentration_numeric"] = metadata["Concentration (ng)"].apply(parse_value)
metadata = compute_alpha_per_group(metadata, comp_col="Treatment", conc_col="concentration_numeric")
metadata, color_dict = assign_colors(metadata, comp_col="Treatment", conc_col="concentration_numeric")
metadata = metadata.loc[:, ~metadata.columns.str.startswith("Unnamed")]
# remove "20" prefix from Name if present, to match SELECT_NAMES
metadata["Name"] = metadata["Name"].apply(lambda x: x[2:] if isinstance(x, str) and x.startswith("20") else x)
metadata = metadata[metadata["Name"] == SELECT_NAMES]

if metadata.empty:
    raise ValueError(f"No metadata rows match SELECT_NAMES='{SELECT_NAMES}' for date {date}.")
print(metadata["Treatment"].unique())
print(f"  {len(metadata)} metadata rows loaded.")
print(metadata.head())

ctrl_rows = metadata[metadata.Treatment.str.contains(r"control|ctrl|dmso|dms0", regex=True, case=False, na=False)]
if ctrl_rows.empty:
    raise ValueError("No control treatment found in metadata (looked for 'control' in Treatment column).")
print(f"  Control treatments found: {ctrl_rows['treatment_full'].tolist()}")
CONTROL_LABEL = ctrl_rows["treatment_full"].values[0]
print(f"  Control label: {CONTROL_LABEL}")

metadata_cols = [
    "Image_Metadata_Well", "treatment_full", "Treatment",
    "Concentration (ng)", "Incub time [h]"
]

# =============================================================================
# CONNECT TO SQLITE
# =============================================================================
print("\n[2/8] Connecting to SQLite...")
conn = sqlite3.connect(SQLITE_FILE)
table_names = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(f"  Tables: {table_names['name'].tolist()}")

# =============================================================================
# PER-IMAGE METADATA + OUTLIER DETECTION
# =============================================================================
print("\n[3/8] Loading Per_Image metadata and detecting outliers...")
per_image = pd.read_sql(
    "SELECT ImageNumber, Image_Metadata_Well, Image_Count_Cells, Mean_Cells_AreaShape_Area "
    "FROM Per_Image",
    conn
)

Experiment : /Volumes/KINGSTON/Nico_data/Cellpaint/20260423_HUVEC_TNF
Root       : /Volumes/KINGSTON/Nico_data/Cellpaint
SQLite     : /Volumes/KINGSTON/Nico_data/Cellpaint/20260423_HUVEC_TNF/cell_profiler_out/CellPaint.db
Metadata   : ./00_Metadata_Experiments.xlsx
Plots  →   : /Volumes/KINGSTON/Nico_data/Cellpaint/20260423_HUVEC_TNF/plt_analysis
Results →  : /Volumes/KINGSTON/Nico_data/Cellpaint/20260423_HUVEC_TNF/rslt

[1/8] Loading metadata...
['TNF ' 'TNF' 'Control' 'Lantrunculin B' 'Sorbitol' 'Cytochalasin D']
  96 metadata rows loaded.
          Date              Name  Well_Column Well_Row Plate Treatment  \
1152  260423.0  260423_HUVEC_TNF          1.0        A    13      TNF    
1153  260423.0  260423_HUVEC_TNF          1.0        B    13      TNF    
1154  260423.0  260423_HUVEC_TNF          1.0        C    13       TNF   
1155  260423.0  260423_HUVEC_TNF          1.0        D    13       TNF   
1156  260423.0  260423_HUVEC_TNF          1.0        E    13       TNF   

     Co

In [6]:
per_image = pd.read_sql("SELECT * FROM Per_Image WHERE ImageNumber = 1", conn)

In [10]:
per_image.keys()

Index(['ImageNumber', 'Image_Count_Cells', 'Image_Count_Cytoplasm',
       'Image_Count_Nuclei', 'Image_ExecutionTime_01Images',
       'Image_ExecutionTime_02Metadata', 'Image_ExecutionTime_03NamesAndTypes',
       'Image_ExecutionTime_04Groups',
       'Image_ExecutionTime_05MeasureImageQuality',
       'Image_ExecutionTime_06MeasureImageQuality',
       ...
       'Mean_Nuclei_Texture_Variance_OrigER_3_02_256',
       'Mean_Nuclei_Texture_Variance_OrigER_3_03_256',
       'Mean_Nuclei_Texture_Variance_OrigMito_3_00_256',
       'Mean_Nuclei_Texture_Variance_OrigMito_3_01_256',
       'Mean_Nuclei_Texture_Variance_OrigMito_3_02_256',
       'Mean_Nuclei_Texture_Variance_OrigMito_3_03_256',
       'Mean_Nuclei_Texture_Variance_OrigRNA_3_00_256',
       'Mean_Nuclei_Texture_Variance_OrigRNA_3_01_256',
       'Mean_Nuclei_Texture_Variance_OrigRNA_3_02_256',
       'Mean_Nuclei_Texture_Variance_OrigRNA_3_03_256'],
      dtype='object', length=1807)

In [ ]:
# find metadata keys
